<a href="https://colab.research.google.com/github/k-sahaj/DS-projects/blob/main/Diabetes_prediction/android_app_api.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Installing the dependencies:

In [4]:
!pip install fastapi
!pip install uvicorn
!pip install pydantic
!pip install scikit-learn
#!pip install pickle coz no need
!pip install requests
!pip install pypi-json
!pip install pyngrok
!pip install nest-asyncio

In [5]:
from fastapi import FastAPI
from pydantic import BaseModel
import pickle
import json
import uvicorn
from pyngrok import ngrok
from fastapi.middleware.cors import CORSMiddleware
import nest_asyncio

In [6]:
app = FastAPI()

In [7]:
origins = ["*"]
app.add_middleware(
    CORSMiddleware,
    allow_origins = origins,
    allow_credentials = True,
    allow_methods = ["*"],
    allow_headers = ["*"],
)

In [8]:
class model_input(BaseModel):
    Pregnancies : int
    Glucose : int
    BloodPressure : int
    SkinThickness : int
    Insulin : int
    BMI : float
    DiabetesPedigreeFunction : float
    Age : int


In [9]:
#loading the saved model
diabetes_model = pickle.load(open('diabetes_model.pkl', 'rb'))

In [19]:
@app.post('/diabetes_prediction')
def diabetes_predd(input_parameters: model_input):

    preg = input_parameters.Pregnancies
    glu = input_parameters.Glucose
    bp = input_parameters.BloodPressure
    skin = input_parameters.SkinThickness
    insulin = input_parameters.Insulin
    bmi = input_parameters.BMI
    dpf = input_parameters.DiabetesPedigreeFunction
    age = input_parameters.Age

    input_list = [preg, glu, bp, skin, insulin, bmi, dpf, age]

    prediction = diabetes_model.predict([input_list])

    print("Input:", input_list)
    print("Prediction:", prediction)
    print("Prediction type:", type(prediction))

    if prediction[0] == 0:
        return {"prediction": "The person is not diabetic"}
    else:
        return {"prediction": "The person is diabetic"}

In [11]:
#ngrok auth_token here
!ngrok authtoken *********** #paste your own

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [12]:
nest_asyncio.apply()

In [21]:
from pyngrok import ngrok

for tunnel in ngrok.get_tunnels():
    print("Closing:", tunnel.public_url)
    ngrok.disconnect(tunnel.public_url)

ngrok.kill() #killing the old created tunnels

In [22]:
#create ngrok tunnel
ngrok_tunnel = ngrok.connect(8000)
print('Public URL:', ngrok_tunnel.public_url)

Public URL: https://anyway-macaw-delusion.ngrok-free.dev


In [25]:
from uvicorn import Config, Server
# Configure FastAPI server
config = Config(
    app=app,
    host="0.0.0.0",
    port=8000,
    log_level="info"
)

server = Server(config)

In [24]:
!fuser -k 8000/tcp #killing the old server on this port if any

In [26]:
# Start server
await server.serve()

INFO:     Started server process [19445]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     152.59.171.50:0 - "GET / HTTP/1.1" 404 Not Found
INFO:     152.59.171.50:0 - "GET /docs HTTP/1.1" 200 OK
INFO:     152.59.171.50:0 - "GET /openapi.json HTTP/1.1" 200 OK
The person is diabetic
INFO:     152.59.171.50:0 - "POST /diabetes_prediction HTTP/1.1" 200 OK


/tmp/ipykernel_19445/2543609943.py:3: PydanticDeprecatedSince20: The `json` method is deprecated; use `model_dump_json` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  input_data = input_parameters.json()
INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [19445]


In [20]:
sample = [[6,148,72,35,0,33.6,0.627,50]]

pred = diabetes_model.predict(sample)

print(pred)

[1]


NOTE: Although the prediction model is running here perfectly fine, when trying it out in the /docs section with same input data, returns 'null'. IDK why.
Whereas the rest of the thing is success.